In [1]:
pip install notebook

In [12]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine("sqlite:///C:/Users/SHALINI/OneDrive/Desktop/database/sales.db")

In [13]:
query = "SELECT * FROM Ecommerce_Sales_Data_2024_2025 LIMIT 5;"
df = pd.read_sql(query, engine)
print(df)

  order_id  order_date customer_name region       city   category  \
0    10001  2024-10-19  Kashvi Varty  South  Bangalore      Books   
1    10002  2025-08-30   Advik Desai  North      Delhi  Groceries   
2    10003  2023-11-04    Rhea Kalla   East      Patna    Kitchen   
3    10004  2025-05-23     Anika Sen   East    Kolkata  Groceries   
4    10005  2025-01-19   Akarsh Kaul   West       Pune   Clothing   

  sub_category       product_name quantity unit_price discount_percent  \
0  Non-Fiction  Non-Fiction Ipsum        2      36294                5   
1         Rice          Rice Nemo        1      42165               20   
2       Juicer        Juicer Odio        4      64876               20   
3          Oil      Oil Doloribus        5      37320               15   
4    Kids Wear      Kids Wear Quo        1      50037               10   

      sales    profit payment_mode  
0   68958.6  10525.09   Debit Card  
1   33732.0   6299.66   Debit Card  
2  207603.2  19850.27  Credit

In [14]:
# 1. Top 5 products by sales
q1 = """
SELECT product_name, SUM(sales) AS total_sales
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY product_name
ORDER BY total_sales DESC LIMIT 5;
"""

# 2. Monthly sales trend
q2 = """
SELECT strftime('%Y-%m', order_date) AS month, SUM(sales) AS monthly_sales
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY month
ORDER BY month;
"""

# 3. Customer segmentation by spend
q3 = """
SELECT customer_name,
       SUM(sales) AS total_spend,
       CASE
           WHEN SUM(sales) > 100000 THEN 'High Value'
           WHEN SUM(sales) > 50000 THEN 'Medium Value'
           ELSE 'Low Value'
       END AS segment
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY customer_name;
"""

# 4. Best performing category by profit
q4 = """
SELECT category, SUM(profit) AS total_profit
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY category
ORDER BY total_profit DESC;
"""

# 5. Average order value by region
q5 = """
SELECT region, AVG(sales) AS avg_order_value
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY region;
"""

# 6. Number of orders per customer
q6 = """
SELECT customer_name, COUNT(order_id) AS num_orders
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY customer_name
ORDER BY num_orders DESC;
"""

# 7. Repeat customers (more than 1 order)
q7 = """
SELECT customer_name, COUNT(order_id) AS orders
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY customer_name
HAVING orders > 1;
"""

# 8. Highest single-order profit
q8 = "SELECT * FROM Ecommerce_Sales_Data_2024_2025 ORDER BY profit DESC LIMIT 1;"

# 9. Sales by payment mode
q9 = """
SELECT payment_mode, SUM(sales) AS total_sales, COUNT(order_id) AS num_orders
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY payment_mode
ORDER BY total_sales DESC;
"""

# 10. Top city by sales in each region
q10 = """
SELECT region, city, SUM(sales) AS city_sales
FROM Ecommerce_Sales_Data_2024_2025
GROUP BY region, city
ORDER BY region, city_sales DESC;
"""

questions = ['Top 5 Products','Monthly Trend','Customer Segmentation','Category Profit',
             'Avg Order Value by Region','Orders per Customer','Repeat Customers',
             'Highest Profit Order','Sales by Payment Mode','Top City per Region']
queries = [q1,q2,q3,q4,q5,q6,q7,q8,q9,q10]

for name, q in zip(questions, queries):
    print(f"\n--- {name} ---")
    result = pd.read_sql(q, engine)
    print(result)


--- Top 5 Products ---
              product_name  total_sales
0   Headphones Accusantium    857184.20
1         Spices Quibusdam    687651.25
2  Accessories Repellendus    687083.30
3              Bed Tenetur    670994.80
4         Laptop Similique    666640.60

--- Monthly Trend ---
      month  monthly_sales
0   2023-10    21307522.20
1   2023-11    22040269.20
2   2023-12    20624240.25
3   2024-01    21477241.40
4   2024-02    19853396.50
5   2024-03    21571015.35
6   2024-04    22214781.10
7   2024-05    24744786.55
8   2024-06    21368962.50
9   2024-07    24019283.90
10  2024-08    23299894.50
11  2024-09    22480182.05
12  2024-10    23607626.65
13  2024-11    22036247.55
14  2024-12    24806786.40
15  2025-01    21520801.80
16  2025-02    19902446.10
17  2025-03    22602988.75
18  2025-04    21653817.90
19  2025-05    26010928.65
20  2025-06    21155496.20
21  2025-07    22526567.55
22  2025-08    23317916.20
23  2025-09    18131496.70
24  2025-10     1391328.40

--- Custom

In [15]:
from sqlalchemy import create_engine
import pandas as pd

def get_engine(db_path='sales.db'):
    """Create a connection engine to the SQLite database."""
    return create_engine(f'sqlite:///{db_path}')

def run_query(query, db_path='sales.db'):
    """Run any SQL query and return results as a DataFrame."""
    engine = get_engine(db_path)
    return pd.read_sql(query, engine)

def load_csv_to_db(csv_path, table_name, db_path='sales.db'):
    """Load a CSV file into the database as a new table."""
    engine = get_engine(db_path)
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"Loaded {len(df)} rows into '{table_name}'")